In [156]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
from torch.utils.data import DataLoader, Dataset, TensorDataset
from diffusers import DDPMScheduler
from tqdm import trange, tqdm
import numpy as np
import random
import os
import math

In [157]:
seed = 42
torch.cuda.manual_seed(seed)
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
cudnn.deterministic = True
cudnn.benchmark = False

In [158]:
DEVICE = torch.device("cuda:1")
# DATA_DIR = '../data/MSLR-Web10K/Fold1/'
DATA_DIR = '../data/MQ2007/Fold1/'

MODE = 'pointwise'

FEAT_COUNT = 136 if 'MSLR' in DATA_DIR else 46
FEAT_SCALE = 1000
MB_SIZE = 1024
NUM_HIDDEN_NODES = 256 if 'MSLR' in DATA_DIR else 128
# NUM_HIDDEN_NODES = 1024
EPOCH_SIZE = 2048
NUM_EPOCHS = 50
LEARNING_RATE = 2e-4
# LEARNING_RATE = 5e-4
DROPOUT_RATE = 0.0


DATA_FILE_TRAIN = os.path.join(DATA_DIR, 'train.txt')
DATA_FILE_TEST = os.path.join(DATA_DIR, 'test.txt')
MODEL_SAVE_PATH = f"ltr.{DATA_DIR.split('/')[-3]}.{NUM_EPOCHS}.pth"

In [159]:
def parse_line(line):
    tokens = line.strip().split(' ')
    qid = -1
    feat = []
    label = int(tokens[0])
    
    for i in range(FEAT_COUNT):
        feat.append(0)
    
    for i in range(1, len(tokens)):
        sub_tokens = tokens[i].split(':')
        if sub_tokens[0] == 'qid':
            qid = int(sub_tokens[1])
        else:
            try:
                feat_idx = int(sub_tokens[0])
                feat_val = float(sub_tokens[1])
                feat[feat_idx - 1] = feat_val
            except:
                pass
    return qid, label, feat

In [160]:
class DataReaderT():
    def __init__(self, data_file):
        self.data_file = data_file

    def __iter__(self):
        self.reader = open(self.data_file, mode='r', encoding="utf-8")
        self.__allocate_minibatch()
        return self
    
    def __allocate_minibatch(self):
        self.features = np.zeros((MB_SIZE, FEAT_COUNT), dtype=np.float32)
        self.labels = np.zeros((MB_SIZE), dtype=np.int64)
        
    def __clear_minibatch(self):
        self.features.fill(np.float32(0))
            
    def __next__(self):
        self.__clear_minibatch()
        qids = []
        labels = []
        cnt = 0
        for i in range(MB_SIZE):
            line = self.reader.readline()
            if line == '':
                raise StopIteration
                break
            
            qid, label, feat = parse_line(line)
            qids.append(qid)
            labels.append(label)
            
            for j in range(FEAT_COUNT):
                self.features[i, j] = feat[j]
            
            cnt += 1
        
        return torch.from_numpy(self.features).to(DEVICE), qids, labels, cnt

In [161]:
# Custom dataset for your features and labels
class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        # Concatenating features and labels to form the input vector
        feature_label_concat = torch.cat((self.features[idx], self.labels[idx].unsqueeze(0)), dim=0)
        return feature_label_concat

# MLP-based diffusion model for 1D feature vectors
class MLPDiffusionModel(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(MLPDiffusionModel, self).__init__()
        self.main = nn.Sequential(
            nn.Linear(input_dim + 1, hidden_dim),
            nn.ReLU(),
            nn.Dropout(DROPOUT_RATE),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(DROPOUT_RATE),
            nn.Linear(hidden_dim, input_dim)  # Output should be the same size as input (features + label)
        )

    def forward(self, x, t):
        input_tensor = torch.cat((x, t), dim=1)
        return self.main(input_tensor)

In [162]:
class WeightedMSELoss(nn.Module):
    def __init__(self, weight):
        super(WeightedMSELoss, self).__init__()
        self.weight = weight

    def forward(self, input, target):
        # Create a weight tensor with the same shape as input/target
        # Set all weights to 1, except for the last feature which is multiplied by the specified weight
        weights = torch.ones_like(input)
        weights[:, -1] = self.weight

        # Compute the squared difference
        squared_diff = (input - target) ** 2

        # Apply the weights and take the mean
        weighted_loss = weights * squared_diff
        return weighted_loss.mean()

In [163]:
READER_TRAIN = DataReaderT(DATA_FILE_TRAIN)
READER_TRAIN_ITER = iter(READER_TRAIN)
READER_TEST = DataReaderT(DATA_FILE_TEST)
READER_TEST_ITER = iter(READER_TEST)

# Initialize the MLP diffusion model
model = MLPDiffusionModel(input_dim=FEAT_COUNT+1, hidden_dim=NUM_HIDDEN_NODES).to(DEVICE)

# Define the noise scheduler
noise_scheduler = DDPMScheduler(num_train_timesteps=1000)

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# criterion = nn.MSELoss()
criterion = WeightedMSELoss(weight=20)

In [164]:
def train_model(data_reader, model, noise_scheduler):
    train_loss = 0.0
    e_size = 0
    for features, _, labels, _ in data_reader:
        e_size += 1
        labels = torch.tensor(labels).to(DEVICE)
        labels = labels.unsqueeze(1)
        data = torch.cat((features, labels), dim=1).to(DEVICE)
        
        # Sample random noise
        noise = torch.randn_like(data)
        
        # Sample random time step t for each example
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (data.shape[0],), device=DEVICE).long().unsqueeze(1)
        
        # Add noise to data based on the timestep
        noisy_data = noise_scheduler.add_noise(data, noise, timesteps)

        # Predict the noise to remove using the MLP
        noise_pred = model(noisy_data, timesteps)

        # Loss: MSE between the added noise and the predicted noise
        loss = criterion(noise_pred, noise)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
    
    return train_loss / e_size


# Define a function for model inference
def inference(features, model, noise_scheduler, num_inference_steps=1000):
    model.eval()

    with torch.no_grad():
        random_labels = torch.randint(0, 5 if 'MSLR' in DATA_DIR else 3, (len(features), 1)).to(DEVICE)
        input_vectors = torch.cat((features, random_labels), dim=1).to(DEVICE)
        input_vectors_dataset = TensorDataset(input_vectors)
        input_vectors_loader = DataLoader(input_vectors_dataset, batch_size=512, shuffle=False)
        
        predictions = []        
        for batch in input_vectors_loader:
            data = batch[0].to(DEVICE)

            # Now denoise from the max timestep (t_max) down to t=0
            for t in reversed(range(num_inference_steps)):
                # Predict the noise at this step
                timesteps = torch.tensor([t]).repeat_interleave(data.shape[0], dim=0).long().to(DEVICE).unsqueeze(1)
                
                if t == num_inference_steps - 1:
                    noise = torch.randn_like(data)
                    noisy_input = noise_scheduler.add_noise(data, noise, timesteps)
                
                noise_pred = model(noisy_input, timesteps)
                
                # Use the scheduler to reverse the noise and move one step back
                noisy_input = noise_scheduler.step(noise_pred, t, noisy_input)['prev_sample']

            # Extract the predicted label (the last element of the denoised vector)
            predicted_label = noisy_input[:, -1]  # Get the predicted label from the denoised vector
            predictions.extend(predicted_label)

        return torch.tensor(predictions).to(DEVICE)


# Function for evaluation
def evaluate_model(data_reader, model, noise_scheduler, num_inference_steps=1000, epoch=0, train_loss=0):
    model.eval()
    
    results = {}
    for features, qids, labels, _ in data_reader:
        # Get predictions from the model
        predicted_labels = inference(features, model, noise_scheduler, num_inference_steps)
        predicted_labels = predicted_labels.cpu()
        for i in range(len(qids)):
            if qids[i] not in results:
                results[qids[i]] = []
            results[qids[i]].append((labels[i], predicted_labels[i]))
    
    avgp = 0
    avgndcg = 0
    for qid, docs in results.items():
        dcg = 0
        ranked = sorted(docs, key=lambda x: x[1], reverse=True)

        relevant_in_top10 = sum(1 for doc in ranked[:10] if doc[0] > 0)
        p = relevant_in_top10 / min(10, len(ranked))
        avgp += p

        for i in range(min(10, len(ranked))):
            rank = i + 1
            label = ranked[i][0]
            dcg += ((2**label - 1) / math.log2(rank + 1))
        idcg = 0
        ranked = sorted(docs, key=lambda x: x[0], reverse=True)
        for i in range(min(10, len(ranked))):
            rank = i + 1
            label = ranked[i][0]
            idcg += ((2**label - 1) / math.log2(rank + 1))
        if idcg > 0:
            avgndcg += (dcg / idcg)
    avgp /= len(results)
    avgndcg /= len(results)
    print(f'epoch:{epoch}, loss: {train_loss}, avgp: {avgp}, avgndcg: {avgndcg}')
    
    return avgp, avgndcg


In [165]:
print('Dataset: {}'.format(DATA_DIR))
print(f"Model has {(sum(p.numel() for p in model.parameters() if p.requires_grad)):,} trainable parameters")
print('Learning rate: {}'.format(LEARNING_RATE))
print('Mode: {} - Diffusers'.format(MODE))

model.train()

train_loss = 'n/a'
for epoch in range(NUM_EPOCHS+1):
    evaluate_model(READER_TEST_ITER, model, noise_scheduler, epoch=epoch, train_loss=train_loss) 
    
    if epoch < NUM_EPOCHS:
        train_loss = train_model(READER_TRAIN_ITER, model, noise_scheduler)


Dataset: ../data/MQ2007/Fold1/
Model has 28,847 trainable parameters
Learning rate: 0.0002
Mode: pointwise - Diffusers
epoch:0, loss: n/a, avgp: 0.28750000000000003, avgndcg: 0.2748132683451932
epoch:1, loss: 13.032023796221106, avgp: 0.27743902439024404, avgndcg: 0.2695330595094146
epoch:2, loss: 1.4624251941355264, avgp: 0.27987804878048783, avgndcg: 0.2725980899376
epoch:3, loss: 1.372124680658666, avgp: 0.2774390243902439, avgndcg: 0.27858129742418336
epoch:4, loss: 1.3479227612658244, avgp: 0.27286585365853644, avgndcg: 0.2641037201495303
epoch:5, loss: 1.3228947244039395, avgp: 0.2807926829268292, avgndcg: 0.27230530820197013
epoch:6, loss: 1.2923252873304414, avgp: 0.27682926829268273, avgndcg: 0.27772276656280187
epoch:7, loss: 1.268850780114895, avgp: 0.2850609756097559, avgndcg: 0.27929512687437735
epoch:8, loss: 1.238796469641895, avgp: 0.27713414634146344, avgndcg: 0.2671778150714705
epoch:9, loss: 1.2093727646804437, avgp: 0.28567073170731705, avgndcg: 0.2832052894038832
e

KeyboardInterrupt: 